# 🏭 Machine Failure Prediction with AI4I 2020: Imbalance, PR/ROC Curves and Cost-Based Threshold

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samtzai/course_ai_for_industrial_control_students/blob/main/notebooks/02_predictive_maintenance/01_supervised_classification/n03b_ai4i_failure_classification.ipynb)

**Part 2 of P4a.** In n03 we classified Iris flowers: a clean, balanced dataset. Real predictive maintenance looks different:
**failures are rare** and **missing a failure is much more expensive than a false alarm**.

We use the **AI4I 2020 Predictive Maintenance Dataset** (S. Matzka, UCI Machine Learning Repository, id 601): 10,000 records of a milling machine with
product type, temperatures, rotational speed, torque and tool wear. Only **≈3.4 %** of the records end in a **machine failure**.

What you will do:
1. Train a PyTorch classifier for `Machine failure` **without leakage** (no failure-mode columns, no IDs).
2. Handle the **class imbalance** with `pos_weight` and see why **accuracy is misleading**.
3. Compare models with **ROC** and **Precision–Recall** curves.
4. **Choose the decision threshold that minimises an economic cost** where a missed failure (FN) costs much more than an unnecessary stop (FP).

📚 Theory: threshold, confusion matrix and metrics (Parte 2, slides 56–67); ROC curve and comparing models (Parte 2, slides 68–69); train/validation/test (Parte 2, slide 52); normalisation (Parte 2, slides 100–102).

In [ ]:
# ⚙️ Setup cell (run first). Detects Google Colab; locally it does nothing.
import sys
IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab' if IN_COLAB else 'Running locally')

In [ ]:
# 📦 Import Required Libraries
import os
import io
import copy
import time
import zipfile
import urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
                             confusion_matrix, precision_score, recall_score, f1_score, accuracy_score)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')   # a CPU is more than enough here
print(f'PyTorch {torch.__version__} | device: {device}')

QUICK_RUN = os.environ.get('QUICK_RUN', '0') == '1'   # only for automatic tests
EPOCHS = 20 if QUICK_RUN else 100
BATCH_SIZE = 256
LEARNING_RATE = 1e-3

# 💶 Economic cost of each type of error (change them to your own scenario!)
COST_FN = 10_000   # missed failure: unplanned stop, broken tool / part, scrap, repair (€)
COST_FP = 500      # false alarm: unnecessary stop and inspection by maintenance (€)
USE_ENGINEERED_FEATURES = False   # try True: adds physically meaningful features (see below)

## 📥 Download the dataset into ./data
Public sources, no login needed: UCI zip → UCI direct CSV → Kaggle mirror via `kagglehub`.

In [ ]:
DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True)
CSV_PATH = DATA_DIR / 'ai4i2020.csv'
AI4I_SOURCES = [
    'https://archive.ics.uci.edu/static/public/601/ai4i+2020+predictive+maintenance+dataset.zip',
    'https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv',
]


def download_ai4i():
    if CSV_PATH.exists():
        return CSV_PATH
    errors = []
    for url in AI4I_SOURCES:
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                payload = r.read()
            if url.endswith('.zip'):
                with zipfile.ZipFile(io.BytesIO(payload)) as zf:
                    name = next(n for n in zf.namelist() if n.endswith('.csv'))
                    payload = zf.read(name)
            CSV_PATH.write_bytes(payload)
            return CSV_PATH
        except Exception as e:  # noqa: BLE001
            errors.append(f'{url}: {e}')
    try:
        os.environ.setdefault('KAGGLEHUB_CACHE', str((DATA_DIR / 'kagglehub').resolve()))
        import kagglehub
        path = Path(kagglehub.dataset_download('stephanmatzka/predictive-maintenance-dataset-ai4i-2020'))
        CSV_PATH.write_bytes(next(path.rglob('*.csv')).read_bytes())
        return CSV_PATH
    except Exception as e:  # noqa: BLE001
        errors.append(f'kagglehub: {e}')
    raise RuntimeError('Could not download AI4I 2020:\n' + '\n'.join(errors) +
                       '\nDownload ai4i2020.csv manually from https://archive.ics.uci.edu/dataset/601 into ./data')


df = pd.read_csv(download_ai4i())
print(df.shape)
df.head()

## 🔎 Explore the data

Columns:
- `UDI`, `Product ID`: **identifiers** → not physical information, we drop them.
- `Type`: product quality variant L / M / H (low / medium / high).
- `Air temperature [K]`, `Process temperature [K]`, `Rotational speed [rpm]`, `Torque [Nm]`, `Tool wear [min]`: **sensor / process variables** → our inputs.
- `Machine failure`: **target** (1 = failure).
- `TWF`, `HDF`, `PWF`, `OSF`, `RNF`: which failure mode happened (tool wear, heat dissipation, power, overstrain, random).

⚠️ **Data leakage:** the failure-mode columns are known only *after* the failure happens, and `Machine failure` is basically their logical OR.
If we used them as inputs, the model would get ~100 % accuracy and would be useless in production. We **must exclude them**.

In [ ]:
TARGET = 'Machine failure'
FAILURE_MODES = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
print(f'Failure rate: {df[TARGET].mean() * 100:.2f} %  ({df[TARGET].sum()} failures out of {len(df)})')
print('\nFailures per mode:'); print(df[FAILURE_MODES].sum())

any_mode = df[FAILURE_MODES].max(axis=1)
print('\nMachine failure vs. "any failure mode" (shows why these columns leak the target):')
print(pd.crosstab(df[TARGET], any_mode, rownames=['Machine failure'], colnames=['any mode = 1']))

fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
for ax, col in zip(axes, ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']):
    for lab, color in [(0, 'tab:blue'), (1, 'tab:red')]:
        ax.hist(df.loc[df[TARGET] == lab, col], bins=40, density=True, alpha=0.5, color=color,
                label='failure' if lab else 'no failure')
    ax.set_title(col, fontsize=9)
axes[0].legend()
plt.suptitle('Distribution of each variable (normalised histograms)')
plt.tight_layout()
plt.show()

## 🧪 Features, split and normalisation

- **Inputs:** `Type` (one-hot) + the 5 process variables. Optional engineered features (`USE_ENGINEERED_FEATURES=True`):
  temperature difference (heat dissipation), mechanical power = torque · ω (power failure) and tool wear · torque (overstrain).
- **Split:** stratified 60 % train / 20 % validation / 20 % test with a fixed seed. The **validation** set is used to choose the threshold; the **test** set only to report the final result.
- **StandardScaler** is fitted **only on train** (fitting it on all the data would leak information from validation/test).

In [ ]:
X_df = pd.get_dummies(df.drop(columns=['UDI', 'Product ID', TARGET] + FAILURE_MODES), columns=['Type'], dtype=float)
if USE_ENGINEERED_FEATURES:
    X_df['Temp diff [K]'] = df['Process temperature [K]'] - df['Air temperature [K]']
    X_df['Power [W]'] = df['Torque [Nm]'] * df['Rotational speed [rpm]'] * 2 * np.pi / 60
    X_df['Wear x Torque [min Nm]'] = df['Tool wear [min]'] * df['Torque [Nm]']
feature_names = list(X_df.columns)
X, y = X_df.values.astype(np.float32), df[TARGET].values.astype(np.float32)
print('Input features:', feature_names)

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=SEED)
scaler = StandardScaler().fit(X_train)
X_train, X_val, X_test = (scaler.transform(a).astype(np.float32) for a in (X_train, X_val, X_test))
for name, t in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(f'{name:>5}: {len(t)} rows | failures = {int(t.sum())} ({100 * t.mean():.2f} %)')

print(f'\n💡 A "model" that ALWAYS predicts "no failure" gets accuracy = {1 - y_test.mean():.3f} on test... and detects 0 failures.')

## 🧠 PyTorch classifier (MLP) and training loop

A small MLP with one output logit and `BCEWithLogitsLoss`. We train **two versions**:
- **A – no weighting**: every sample counts the same.
- **B – `pos_weight = n_negative / n_positive`** (≈ 28): each failure counts as much as ~28 normal samples in the loss.

We keep the weights of the epoch with the best validation loss (early stopping).

In [ ]:
class FailureMLP(nn.Module):
    def __init__(self, n_in, hidden=(32, 16), dropout=0.1):
        super().__init__()
        layers, prev = [], n_in
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)


def to_loader(Xa, ya, shuffle):
    return DataLoader(TensorDataset(torch.from_numpy(Xa), torch.from_numpy(ya)), batch_size=BATCH_SIZE, shuffle=shuffle)


train_loader, val_loader = to_loader(X_train, y_train, True), to_loader(X_val, y_val, False)


def train_classifier(pos_weight=None, name=''):
    torch.manual_seed(SEED)
    model = FailureMLP(X_train.shape[1]).to(device)
    pw = None if pos_weight is None else torch.tensor([pos_weight], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    hist, best, best_state, start = {'train': [], 'val': []}, np.inf, None, time.time()
    for _epoch in range(EPOCHS):
        model.train()
        tot = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            tot += loss.item() * len(yb)
        model.eval()
        with torch.no_grad():
            vl = sum(criterion(model(xb.to(device)), yb.to(device)).item() * len(yb) for xb, yb in val_loader) / len(y_val)
        hist['train'].append(tot / len(y_train)); hist['val'].append(vl)
        if vl < best:
            best, best_state = vl, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    print(f'{name}: trained {EPOCHS} epochs in {time.time() - start:.1f} s, best val loss {best:.4f}')
    return model, hist


@torch.no_grad()
def predict_proba(model, Xa):
    model.eval()
    return torch.sigmoid(model(torch.from_numpy(Xa).to(device))).cpu().numpy()


n_pos = y_train.sum()
POS_WEIGHT = float((len(y_train) - n_pos) / n_pos)
model_a, hist_a = train_classifier(None, 'A - no weighting')
model_b, hist_b = train_classifier(POS_WEIGHT, f'B - pos_weight={POS_WEIGHT:.1f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, h, t in [(axes[0], hist_a, 'A - no weighting'), (axes[1], hist_b, 'B - pos_weight')]:
    ax.plot(h['train'], label='train'); ax.plot(h['val'], label='validation')
    ax.set_title(f'Loss - {t}'); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 📊 Accuracy is not enough: metrics at threshold 0.5 and ROC / PR curves (validation)

In [ ]:
p_val = {'A - no weighting': predict_proba(model_a, X_val), 'B - pos_weight': predict_proba(model_b, X_val)}
p_test = {'A - no weighting': predict_proba(model_a, X_test), 'B - pos_weight': predict_proba(model_b, X_test)}

rows = []
for name, p in p_val.items():
    pred = (p >= 0.5).astype(int)
    rows.append({'model': name, 'accuracy': accuracy_score(y_val, pred), 'precision': precision_score(y_val, pred, zero_division=0),
                 'recall': recall_score(y_val, pred), 'F1': f1_score(y_val, pred), 'ROC AUC': roc_auc_score(y_val, p),
                 'PR AUC (AP)': average_precision_score(y_val, p)})
print('Validation metrics at threshold 0.5:')
print(pd.DataFrame(rows).round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, p in p_val.items():
    fpr, tpr, _ = roc_curve(y_val, p)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_val, p):.3f})')
    prec, rec, _ = precision_recall_curve(y_val, p)
    axes[1].plot(rec, prec, label=f'{name} (AP={average_precision_score(y_val, p):.3f})')
axes[0].plot([0, 1], [0, 1], 'k:', label='random'); axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate (recall)')
axes[0].set_title('ROC curve (validation)')
axes[1].axhline(y_val.mean(), color='k', ls=':', label=f'random (= failure rate {y_val.mean():.3f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].set_title('Precision-Recall curve (validation)')
for ax in axes:
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

💡 With only ~3 % positives, the ROC curve can look excellent even when precision is poor, because the false-positive *rate* is divided by a huge number of negatives.
The **PR curve** shows directly how many of the alarms are real failures.

## 💶 Choose the threshold that minimises the cost

For each threshold `t` we compute, on the **validation** set:

$$\text{Cost}(t) = FN(t)\cdot C_{FN} + FP(t)\cdot C_{FP}$$

and we pick the `t` with the lowest cost. Then we apply that **fixed** threshold to the test set.
We also show the threshold that maximises F1 (a typical choice, Parte 2 slide 67) and the default 0.5.

In [ ]:
THRESHOLDS = np.linspace(0.001, 0.999, 999)


def cost_curve(y_true, p, c_fn=None, c_fp=None):
    c_fn = COST_FN if c_fn is None else c_fn
    c_fp = COST_FP if c_fp is None else c_fp
    pos = y_true == 1
    fn = np.array([np.sum(pos & (p < t)) for t in THRESHOLDS])
    fp = np.array([np.sum(~pos & (p >= t)) for t in THRESHOLDS])
    return fn * c_fn + fp * c_fp, fn, fp


MODEL_FOR_DECISION = 'B - pos_weight'   # change to 'A - no weighting' and compare
pv = p_val[MODEL_FOR_DECISION]
cost_val, fn_val, fp_val = cost_curve(y_val, pv)
t_cost = float(THRESHOLDS[np.argmin(cost_val)])
f1s = [f1_score(y_val, (pv >= t).astype(int), zero_division=0) for t in THRESHOLDS]
t_f1 = float(THRESHOLDS[int(np.argmax(f1s))])
print(f'Model: {MODEL_FOR_DECISION} | C_FN = {COST_FN} €, C_FP = {COST_FP} € (ratio {COST_FN / COST_FP:.0f}:1)')
print(f'Cost-optimal threshold (validation): {t_cost:.3f} | F1-optimal threshold: {t_f1:.3f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(THRESHOLDS, cost_val / 1000, label='total cost (validation)')
ax.plot(THRESHOLDS, fn_val * COST_FN / 1000, '--', label='cost of FN')
ax.plot(THRESHOLDS, fp_val * COST_FP / 1000, '--', label='cost of FP')
for t, lab, c in [(0.5, 'default 0.5', 'gray'), (t_f1, 'max F1', 'green'), (t_cost, 'min cost', 'red')]:
    ax.axvline(t, color=c, ls=':', label=f'{lab} (t={t:.2f})')
ax.set_xlabel('Decision threshold'); ax.set_ylabel('Cost (k€)'); ax.set_xscale('logit')
ax.set_title('Cost vs threshold (validation)'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 🏁 Final evaluation on the test set (thresholds fixed on validation)

In [ ]:
pt = p_test[MODEL_FOR_DECISION]
n_test = len(y_test)
summary = []
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, (lab, t) in zip(axes, [('default 0.5', 0.5), ('max F1 (val)', t_f1), ('min cost (val)', t_cost)]):
    pred = (pt >= t).astype(int)
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()
    cost = fn * COST_FN + fp * COST_FP
    summary.append({'threshold': lab, 't': round(t, 3), 'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn,
                    'recall': tp / (tp + fn), 'precision': tp / max(tp + fp, 1), 'cost (€)': cost,
                    'cost per 1000 records (€)': 1000 * cost / n_test})
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['no failure', 'failure'], yticklabels=['no failure', 'failure'])
    ax.set_title(f'{lab}: t={t:.3f}\ncost = {cost / 1000:.1f} k€'); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

n_fail = int(y_test.sum())
summary.append({'threshold': 'never intervene', 'TP': 0, 'FN': n_fail, 'FP': 0, 'TN': n_test - n_fail, 'recall': 0.0,
                'cost (€)': n_fail * COST_FN, 'cost per 1000 records (€)': 1000 * n_fail * COST_FN / n_test})
summary.append({'threshold': 'always intervene', 'TP': n_fail, 'FN': 0, 'FP': n_test - n_fail, 'TN': 0, 'recall': 1.0,
                'cost (€)': (n_test - n_fail) * COST_FP, 'cost per 1000 records (€)': 1000 * (n_test - n_fail) * COST_FP / n_test})
print(pd.DataFrame(summary).round(3).to_string(index=False))

## 🎛️ How does the optimal threshold depend on the cost ratio?

If the predicted probabilities were perfectly **calibrated**, decision theory gives the optimal threshold directly:
$t^* = \dfrac{C_{FP}}{C_{FP} + C_{FN}}$. Model A (no weighting) is closer to calibrated; `pos_weight` pushes the probabilities of model B upwards,
so its optimal threshold is higher. Below we repeat the search on validation for different ratios $C_{FN}/C_{FP}$.

In [ ]:
ratios = [1, 2, 5, 10, 20, 50, 100]
fig, ax = plt.subplots(figsize=(7, 4))
for name, p in p_val.items():
    best_t = []
    for r in ratios:
        best_t.append(THRESHOLDS[np.argmin(cost_curve(y_val, p, c_fn=r * COST_FP)[0])])
    ax.plot(ratios, best_t, 'o-', label=name)
ax.plot(ratios, [1 / (1 + r) for r in ratios], 'k--', label='theory (calibrated): 1/(1+ratio)')
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('C_FN / C_FP'); ax.set_ylabel('optimal threshold (validation)')
ax.set_title('Optimal threshold vs cost ratio'); ax.legend(fontsize=8); ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## 📝 Summary

- With rare failures, **accuracy is misleading**: "never failure" already gets ~97 %.
- `pos_weight` changes what the model optimises (more weight on failures) and shifts its probabilities; it does **not** remove the need to choose a threshold.
- **ROC** compares models independently of the threshold; with strong imbalance the **PR curve** is more informative.
- The threshold is a **business decision**: choose it on **validation** with your own costs, then report **test** results once.
- Leakage check: the failure-mode columns and IDs were excluded. Ask yourself for every input: *would I know this value before the failure happens?*

For the quiz you will need **your own** numbers: confusion matrices, thresholds, costs and how they change when you change `COST_FN`, `COST_FP`, `USE_ENGINEERED_FEATURES` or the model used for the decision.